# FinBot Fine-Tuning Notebook

This notebook is rebuilt for safe end-to-end execution in Google Colab.

## 1. Install dependencies

In [ ]:
try:
    import subprocess
    import sys

    print('Installing required packages...')
    subprocess.check_call(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            'transformers',
            'peft',
            'trl',
            'datasets',
            'accelerate',
            'bitsandbytes',
        ]
    )
    print('Dependency installation completed.')
except Exception as exc:
    print(f'[Cell 1] Dependency installation failed: {type(exc).__name__}: {exc}')
    raise


## 2. Mount Drive and set paths

In [ ]:
try:
    import os
    from pathlib import Path

    import torch
    from google.colab import drive
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    drive.mount('/content/drive', force_remount=True)

    if not torch.cuda.is_available():
        raise EnvironmentError('CUDA GPU is required for 4-bit fine-tuning in this notebook.')

    os.environ['TOKENIZERS_PARALLELISM'] = 'false'

    MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
    TRAIN_PATH = '/content/drive/MyDrive/2026/train.jsonl'
    OUTPUT_DIR = '/content/drive/MyDrive/finbot-model'
    CHECKPOINT_DIR = '/content/finbot-checkpoints'

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    if not Path(TRAIN_PATH).is_file():
        raise FileNotFoundError(f'Training file not found: {TRAIN_PATH}')

    print(f'MODEL_NAME={MODEL_NAME}')
    print(f'TRAIN_PATH={TRAIN_PATH}')
    print(f'OUTPUT_DIR={OUTPUT_DIR}')
    print(f'CUDA device={torch.cuda.get_device_name(0)}')
except Exception as exc:
    print(f'[Cell 2] Setup failed: {type(exc).__name__}: {exc}')
    raise


## 3. Load and format dataset

In [ ]:
try:
    from datasets import load_dataset

    if 'TRAIN_PATH' not in globals():
        raise RuntimeError('TRAIN_PATH is not defined. Run Cell 2 first.')

    dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')

    required_columns = {'instruction', 'input', 'output'}
    missing_columns = required_columns.difference(dataset.column_names)
    if missing_columns:
        raise KeyError(f'Missing required dataset columns: {sorted(missing_columns)}')

    original_columns = list(dataset.column_names)

    def format_example(example):
        prompt = (
            '### Instruction:\n'
            f"{example['instruction']}\n\n"
            '### Input:\n'
            f"{example['input']}\n\n"
            '### Response:\n'
            f"{example['output']}"
        )
        return {'text': prompt}

    dataset = dataset.map(format_example, remove_columns=original_columns)

    if len(dataset) == 0:
        raise RuntimeError('Dataset is empty after loading.')
    if 'text' not in dataset.column_names:
        raise RuntimeError("Formatted dataset is missing the 'text' column.")
    if not dataset[0]['text'].strip():
        raise RuntimeError('First formatted sample is empty.')

    print(f'Dataset rows: {len(dataset)}')
    print(dataset[0]['text'][:500])
except Exception as exc:
    print(f'[Cell 3] Dataset preparation failed: {type(exc).__name__}: {exc}')
    raise


## 4. Load tokenizer and base model

In [ ]:
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    required_globals = ['MODEL_NAME']
    missing_globals = [name for name in required_globals if name not in globals()]
    if missing_globals:
        raise RuntimeError(f'Missing prerequisites: {missing_globals}. Run Cell 2 first.')

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False

    print('Tokenizer and model loaded successfully.')
    print(f'Model device map present: {hasattr(model, "hf_device_map")}')
except Exception as exc:
    print(f'[Cell 4] Model loading failed: {type(exc).__name__}: {exc}')
    raise


## 5. Prepare LoRA

In [ ]:
try:
    from peft import LoraConfig, prepare_model_for_kbit_training

    if 'model' not in globals():
        raise RuntimeError('Model is not loaded. Run Cell 4 first.')

    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'v_proj'],
    )

    print('LoRA configuration prepared.')
    print(peft_config)
except Exception as exc:
    print(f'[Cell 5] LoRA preparation failed: {type(exc).__name__}: {exc}')
    raise


## 6. Build trainer

In [ ]:
try:
    import torch
    from trl import SFTConfig, SFTTrainer

    required_globals = ['CHECKPOINT_DIR', 'dataset', 'model', 'peft_config', 'tokenizer']
    missing_globals = [name for name in required_globals if name not in globals()]
    if missing_globals:
        raise RuntimeError(f'Missing prerequisites: {missing_globals}')
    if 'text' not in dataset.column_names:
        raise RuntimeError("Dataset must contain a 'text' column before trainer creation.")
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU is not available at trainer initialization time.')

    training_args = SFTConfig(
        output_dir=CHECKPOINT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy='epoch',
        fp16=True,
        report_to='none',
        optim='paged_adamw_8bit',
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        dataset_text_field='text',
        max_seq_length=1024,
    )

    print('Trainer initialized successfully.')
except Exception as exc:
    print(f'[Cell 6] Trainer initialization failed: {type(exc).__name__}: {exc}')
    raise


## 7. Train

In [ ]:
try:
    if 'trainer' not in globals():
        raise RuntimeError('Trainer is not initialized. Run Cell 6 first.')

    train_result = trainer.train()
    print('Training completed.')
    print(train_result)
except Exception as exc:
    print(f'[Cell 7] Training failed: {type(exc).__name__}: {exc}')
    raise


## 8. Save artifacts

In [ ]:
try:
    import os

    required_globals = ['OUTPUT_DIR', 'tokenizer', 'trainer']
    missing_globals = [name for name in required_globals if name not in globals()]
    if missing_globals:
        raise RuntimeError(f'Missing prerequisites: {missing_globals}')

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f'Saved model and tokenizer to: {OUTPUT_DIR}')
except Exception as exc:
    print(f'[Cell 8] Save failed: {type(exc).__name__}: {exc}')
    raise


## 9. Smoke test generation

In [ ]:
try:
    import torch

    required_globals = ['tokenizer', 'trainer']
    missing_globals = [name for name in required_globals if name not in globals()]
    if missing_globals:
        raise RuntimeError(f'Missing prerequisites: {missing_globals}')

    test_prompt = (
        '### Instruction:\n'
        'Answer the following finance question.\n\n'
        '### Input:\n'
        'Summarize the operating profit trend for fiscal year 2024.\n\n'
        '### Response:\n'
    )

    inference_model = trainer.model
    inference_model.eval()
    model_device = next(inference_model.parameters()).device
    inputs = tokenizer(test_prompt, return_tensors='pt').to(model_device)
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
except Exception as exc:
    print(f'[Cell 9] Inference failed: {type(exc).__name__}: {exc}')
    raise
